# CRAFT Stage 3: Neural Reranking

This notebook implements Stage 3 of the CRAFT pipeline: Neural reranking using OpenAI or Gemini embeddings.

**Pipeline Flow:**
1. Load Stage 2 results (top 100 candidates per query)
2. Apply neural embeddings (OpenAI text-embedding-3 or Gemini)
3. Perform final precision ranking
4. Output final ranked results for answer generation

In [ ]:
# Import CRAFT modular components
from craft_core import CRAFTConfig, DataLoader, TableProcessor, ResultsManager, setup_logging
from craft_stages import create_neural_reranker, setup_nq_pipeline, setup_ottqa_pipeline

import pandas as pd
import json
import numpy as np
from tqdm import tqdm
import logging
import os
from datetime import datetime

# Setup logging
setup_logging("INFO")
logger = logging.getLogger(__name__)

# Configuration
DATASET = "nq"  # "nq" or "ottqa"
USE_GEMINI = DATASET == "ottqa"  # Use Gemini for OTT-QA, OpenAI for NQ

# Initialize CRAFT configuration
config = CRAFTConfig(DATASET)
data_loader = DataLoader()
table_processor = TableProcessor()
results_manager = ResultsManager()

print(f"🚀 CRAFT Stage 3: Neural Reranking")
print(f"📊 Dataset: {DATASET.upper()}")
print(f"🤖 Using: {'Gemini' if USE_GEMINI else 'OpenAI'}")

# Get pipeline configuration
pipeline_config = setup_nq_pipeline() if DATASET == "nq" else setup_ottqa_pipeline()
model_name = pipeline_config['stage3']

logger.info(f"🤖 Using model: {model_name}")

In [ ]:
# Load Stage 2 results and setup neural reranker
logger.info("📂 Loading Stage 2 results...")

# Load Stage 2 results
stage2_results_path = config.base_dir / "datasets" / f"CRAFT_{DATASET.upper()}_2ndStage_Result_dense_reranking.jsonl"
if not stage2_results_path.exists():
    # Try alternative path
    import glob
    search_pattern = str(config.base_dir / "results" / "stage2" / f"*Stage2*.pkl")
    matching_files = glob.glob(search_pattern)
    if matching_files:
        stage2_results_path = matching_files[0]
        stage2_results = data_loader.load_pickle(stage2_results_path)
    else:
        raise FileNotFoundError(f"Stage 2 results not found. Expected at {stage2_results_path}")
else:
    # Load from JSONL
    stage2_results = {}
    with open(stage2_results_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            stage2_results[data['query_id']] = data['stage2_results']

logger.info(f"✅ Loaded Stage 2 results for {len(stage2_results)} queries")

# Setup API key
api_key_env = "GEMINI_API_KEY" if USE_GEMINI else "OPENAI_API_KEY"
api_key = os.getenv(api_key_env)

if not api_key:
    logger.error(f"❌ {api_key_env} environment variable required")
    raise ValueError(f"Please set {api_key_env} environment variable")

# Initialize neural reranker
logger.info("🤖 Initializing neural reranker...")
reranker = create_neural_reranker(
    model_name=model_name,
    api_key=api_key,
    use_gemini=USE_GEMINI
)
logger.info("✅ Neural reranker ready")

In [ ]:
# Load metadata and process queries with neural reranking
logger.info("📋 Loading metadata and questions...")

metadata = data_loader.load_metadata(config.get_path("metadata"))
questions = data_loader.load_questions(config.get_path("questions"))

logger.info(f"🔄 Processing {len(stage2_results)} queries with neural reranking...")

final_results = {}
failed_queries = []
processed_count = 0

for query_id, stage2_candidates in tqdm(stage2_results.items(), desc="Neural reranking"):
    try:
        # Get query information
        if query_id in questions.index.astype(str):
            query_row = questions.loc[questions.index.astype(str) == query_id].iloc[0]
        else:
            query_idx = int(query_id) if query_id.isdigit() else None
            if query_idx is not None and query_idx < len(questions):
                query_row = questions.iloc[query_idx]
            else:
                logger.warning(f"Query {query_id} not found")
                failed_queries.append(query_id)
                continue
        
        question = query_row.get('question', query_row.get('text', ''))
        
        # Extract table IDs from Stage 2 results
        if isinstance(stage2_candidates, list) and stage2_candidates:
            table_ids = [tid for tid, _ in stage2_candidates]
        else:
            logger.warning(f"Invalid Stage 2 format for query {query_id}")
            failed_queries.append(query_id)
            continue
        
        # Get full table texts for neural reranking
        table_texts = []
        valid_table_ids = []
        
        for table_id in table_ids:
            table_info = metadata[metadata['TableID'] == table_id]
            if not table_info.empty:
                table_text = table_processor.create_table_text(table_info.iloc[0])
                table_texts.append(table_text)
                valid_table_ids.append(table_id)
        
        if not table_texts:
            logger.warning(f"No valid tables for query {query_id}")
            failed_queries.append(query_id)
            continue
        
        # Final neural reranking
        final_ranking = reranker.rerank(
            question,
            table_texts,
            valid_table_ids,
            batch_size=20  # Smaller batches for API rate limits
        )
        
        final_results[query_id] = final_ranking
        processed_count += 1
        
        # Progress logging
        if processed_count % 50 == 0:
            logger.info(f"Processed {processed_count} queries...")
            
        # Rate limiting for APIs
        if USE_GEMINI:
            import time
            time.sleep(0.5)  # Respect rate limits
            
    except Exception as e:
        logger.error(f"Error processing query {query_id}: {e}")
        failed_queries.append(query_id)

logger.info(f"✅ Successfully processed {len(final_results)} queries")
if failed_queries:
    logger.warning(f"⚠️ Failed queries: {len(failed_queries)}")

In [ ]:
# Save final results and evaluate
logger.info("💾 Saving Stage 3 final results...")

# Save pickle format
pickle_path = results_manager.save_stage_results(
    final_results, 3, DATASET, config.get_path("stage3_output").parent
)

# Save JSONL format 
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
jsonl_path = config.base_dir / "results" / "stage3" / f"CRAFT_{DATASET.upper()}_Final_Stage_{'Gemini' if USE_GEMINI else 'OpenAI'}_Results_100_reranked_{timestamp}.jsonl"
jsonl_path.parent.mkdir(parents=True, exist_ok=True)

with open(jsonl_path, 'w') as f:
    for query_id, results in final_results.items():
        # Get query details
        if query_id in questions.index.astype(str):
            query_row = questions.loc[questions.index.astype(str) == query_id].iloc[0]
        else:
            query_idx = int(query_id) if query_id.isdigit() else 0
            query_row = questions.iloc[query_idx] if query_idx < len(questions) else {}
        
        question = query_row.get('question', query_row.get('text', ''))
        gold_table_id = query_row.get('gold_table_id', '')
        
        output_data = {
            'query_id': query_id,
            'question': question,
            'gold_table_id': gold_table_id,
            'final_ranking': results[:100],  # Top 100
            'model_used': model_name,
            'timestamp': timestamp
        }
        
        f.write(json.dumps(output_data, ensure_ascii=False) + '\n')

# Evaluate final results
if 'gold_table_id' in questions.columns:
    logger.info("📊 Computing final evaluation metrics...")
    
    from craft_core import EvaluationMetrics
    metrics = EvaluationMetrics()
    
    # Prepare evaluation data
    gold_answers = {}
    rankings = {}
    
    for query_id, results in final_results.items():
        if query_id in questions.index.astype(str):
            query_row = questions.loc[questions.index.astype(str) == query_id].iloc[0]
        else:
            query_idx = int(query_id) if query_id.isdigit() else 0
            query_row = questions.iloc[query_idx] if query_idx < len(questions) else {}
        
        gold_table = query_row.get('gold_table_id', '')
        if gold_table:
            gold_answers[query_id] = gold_table
            rankings[query_id] = [tid for tid, _ in results]
    
    # Compute metrics
    recall_scores = metrics.compute_recall_at_k(rankings, gold_answers, [1, 10, 50])
    mrr_score = metrics.compute_mrr(rankings, gold_answers)
    
    logger.info("🏆 CRAFT Final Pipeline Results:")
    for k, score in recall_scores.items():
        logger.info(f"   Recall@{k}: {score:.2f}%")
    logger.info(f"   MRR: {mrr_score:.4f}")

logger.info(f"✅ Stage 3 results saved to:")
logger.info(f"   Pickle: {pickle_path}")
logger.info(f"   JSONL: {jsonl_path}")

logger.info("🎉 CRAFT 3-Stage Pipeline Complete!")
logger.info(f"📊 Final Summary:")
logger.info(f"   Total queries processed: {len(final_results)}")
logger.info(f"   Model used: {model_name}")
logger.info(f"   API provider: {'Gemini' if USE_GEMINI else 'OpenAI'}")
logger.info(f"   Processing time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")